In [97]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

from sklearn.preprocessing import MinMaxScaler

In [98]:
SEED=42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [99]:
data=pd.read_csv('/content/all_stocks_5yr.csv',on_bad_lines="skip")
data['date']=pd.to_datetime(data['date'])
data=data.sort_values('date')

In [100]:
data.head()

,date,open,high,low,close,volume,Name
0,2013-02-08,15.07,15.1200,14.630,14.75,8407500,AAL
508224,2013-02-08,81.22,81.9300,80.940,81.89,296853,SLG
506965,2013-02-08,78.24,79.0700,78.125,79.07,4632684,SLB
85755,2013-02-08,236.64,238.6924,235.750,238.16,552207,BLK
505706,2013-02-08,89.04,89.4800,88.910,89.16,554948,SJM


In [101]:
apple=data[data['Name']=="AAPL"]
close_data=apple[["close"]].values

In [102]:
train_size=int(len(close_data)*0.8)
val_size=int(len(close_data)*0.1)
train_dataset=close_data[:train_size]
val_dataset=close_data[train_size:train_size+val_size]
test_dataset=close_data[train_size+val_size:]

In [103]:
scaler=MinMaxScaler()
train_data=scaler.fit_transform(train_dataset)
val_data=scaler.transform(val_dataset)
test_data=scaler.transform(test_dataset)

In [104]:
train_data[0]

array([0.15625287])

## Sequencing

In [105]:
def create_seq(data,window_size=60):
  xs,ys=[],[]
  for i in range(window_size,len(data)):
    xs.append(data[i-window_size:i,0])
    ys.append(data[i,0])
  return np.array(xs),np.array(ys)

In [106]:
x_train,y_train=create_seq(train_data)
x_val,y_val=create_seq(val_data)
x_test,y_test=create_seq(test_data)

In [107]:
x_train=torch.tensor(x_train).float().unsqueeze(-1)
y_train=torch.tensor(y_train).float().unsqueeze(-1)
x_val=torch.tensor(x_val).float().unsqueeze(-1)
y_val=torch.tensor(y_val).float().unsqueeze(-1)
x_test=torch.tensor(x_test).float().unsqueeze(-1)
y_test=torch.tensor(y_test).float().unsqueeze(-1)
train_loader=DataLoader(TensorDataset(x_train,y_train),batch_size=28,shuffle=False)
val_loader=DataLoader(TensorDataset(x_val,y_val),batch_size=32,shuffle=False)
test_loader=DataLoader(TensorDataset(x_test,y_test),batch_size=32,shuffle=False)

In [108]:
x_train.shape

torch.Size([947, 60, 1])

In [109]:
class StockLSTM(nn.Module):
  def __init__(self):
    super().__init__()
    self.lstm1 = nn.LSTM(input_size=1, hidden_size=64, batch_first=True)
    self.lstm2 = nn.LSTM(input_size=64, hidden_size=64, batch_first=True)
    self.fc = nn.Sequential(
        nn.Linear(in_features=64, out_features=32),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(in_features=32, out_features=1)
    )

  def forward(self, x):
    out, _ = self.lstm1(x)
    out, _ = self.lstm2(out)
    out = out[:, -1, :]
    return self.fc(out)

In [110]:
model=StockLSTM()
criterion=nn.MSELoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

Traning Loop


In [111]:
Epochs=15
for epoch in range(Epochs):
  model.train()
  for xb,yb in train_loader:
    optimizer.zero_grad()
    preds=model(xb)
    loss=criterion(preds,yb)
    optimizer.step()

model.eval()
val_loss=[]
with torch.inference_mode():
  for xb,yb in val_loader:
    vpred=model(xb)
    val_loss.append(criterion(vpred,yb).item())
print(f"Epoch {epoch+1}/{Epochs}, Val MSE={np.mean(val_loss):.6f}")

Epoch 15/15, Val MSE=1.498736


Testing

In [112]:
model.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        preds.append(model(xb).numpy())

preds = np.vstack(preds)
preds_unscaled = scaler.inverse_transform(preds)
y_test_unscaled = scaler.inverse_transform(y_test.numpy())

mse = np.mean((preds_unscaled - y_test_unscaled)**2)
rmse = np.sqrt(mse)

print("Test MSE:", mse)
print("Test RMSE:", rmse)

Test MSE: 12864.317
Test RMSE: 113.420975
